In [1]:
from pathlib import Path
from PIL import Image, ImageEnhance, ImageFilter
import shutil
import numpy as np
import random

random.seed(42)

SRC = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive2")
DST = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive2_aug")

SRC_TRAIN = SRC / "train"
SRC_VAL = SRC / "val"

DST_TRAIN = DST / "train"
DST_VAL = DST / "val"

DST_TRAIN.mkdir(parents=True, exist_ok=True)
DST_VAL.mkdir(parents=True, exist_ok=True)

print("Source exists:", SRC.exists())
print("Train exists:", SRC_TRAIN.exists())
print("Val exists:", SRC_VAL.exists())
print("Destination ready:", DST.exists())

Source exists: True
Train exists: True
Val exists: True
Destination ready: True


In [2]:
train_classes = [p.name for p in SRC_TRAIN.iterdir() if p.is_dir()]
val_classes = [p.name for p in SRC_VAL.iterdir() if p.is_dir()]

print("Train classes:", train_classes)
print("Val classes:", val_classes)
print("Number of train classes:", len(train_classes))
print("Number of val classes:", len(val_classes))

Train classes: ['01-minor', '02-moderate', '03-severe']
Val classes: ['01-minor', '02-moderate', '03-severe']
Number of train classes: 3
Number of val classes: 3


In [3]:
def aug_brightness(img: Image.Image, factor: float) -> Image.Image:
    return ImageEnhance.Brightness(img).enhance(factor)

def aug_contrast(img: Image.Image, factor: float) -> Image.Image:
    return ImageEnhance.Contrast(img).enhance(factor)

def aug_blur(img: Image.Image, radius: float = 1.5) -> Image.Image:
    return img.filter(ImageFilter.GaussianBlur(radius))

def aug_noise(img: Image.Image, noise_level: int = 12) -> Image.Image:
    arr = np.array(img).astype(np.int16)
    noise = np.random.randint(-noise_level, noise_level + 1, arr.shape, dtype=np.int16)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def aug_shadow(img: Image.Image, factor: float = 0.7) -> Image.Image:
    return ImageEnhance.Brightness(img).enhance(factor)

In [4]:
val_copy_count = 0

for class_dir in SRC_VAL.iterdir():
    if not class_dir.is_dir():
        continue

    dst_class_dir = DST_VAL / class_dir.name
    dst_class_dir.mkdir(parents=True, exist_ok=True)

    for img_path in class_dir.glob("*.*"):
        shutil.copy2(img_path, dst_class_dir / img_path.name)
        val_copy_count += 1

print("Validation images copied:", val_copy_count)

Validation images copied: 248


In [5]:
train_copy_count = 0

for class_dir in SRC_TRAIN.iterdir():
    if not class_dir.is_dir():
        continue

    dst_class_dir = DST_TRAIN / class_dir.name
    dst_class_dir.mkdir(parents=True, exist_ok=True)

    for img_path in class_dir.glob("*.*"):
        shutil.copy2(img_path, dst_class_dir / img_path.name)
        train_copy_count += 1

print("Original training images copied:", train_copy_count)

Original training images copied: 1383


In [6]:
aug_count = 0

for class_dir in SRC_TRAIN.iterdir():
    if not class_dir.is_dir():
        continue

    dst_class_dir = DST_TRAIN / class_dir.name
    dst_class_dir.mkdir(parents=True, exist_ok=True)

    for img_path in class_dir.glob("*.*"):
        stem = img_path.stem
        suffix = img_path.suffix

        img = Image.open(img_path).convert("RGB")

        augmentations = [
            ("bright", aug_brightness(img, 1.25)),
            ("dark", aug_brightness(img, 0.75)),
            ("contrast", aug_contrast(img, 1.3)),
            ("blur", aug_blur(img, 1.5)),
            ("noise", aug_noise(img, 12)),
            ("shadow", aug_shadow(img, 0.65)),
        ]

        for tag, aug_img in augmentations:
            new_name = f"{stem}_{tag}{suffix}"
            aug_img.save(dst_class_dir / new_name)
            aug_count += 1

print("Augmented images created:", aug_count)

Augmented images created: 8298


In [7]:
final_train_count = 0
final_val_count = 0

for class_dir in DST_TRAIN.iterdir():
    if class_dir.is_dir():
        final_train_count += len(list(class_dir.glob("*.*")))

for class_dir in DST_VAL.iterdir():
    if class_dir.is_dir():
        final_val_count += len(list(class_dir.glob("*.*")))

print("Final train image count:", final_train_count)
print("Final val image count:", final_val_count)

Final train image count: 9681
Final val image count: 248
